# 00 · Data Loading & Verification
**AgriShield — Kenya 47 counties, 2015-2025 (6204 county-months)**

Load the three harmonised tables produced by `data_pipeline` and create `ml_base_2015_2025.csv` for the ML pipeline. The panel is the contract: 47 counties × 132 months = 6204 rows, no gaps, no NA, with `_imputed` flags.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../../data_pipeline/data/processed")

# 47 official counties (must match data_pipeline/data/kenya_county_crosswalk.csv)
COUNTIES = ["Baringo","Bomet","Bungoma","Busia","Elgeyo Marakwet","Embu","Garissa","Homa Bay","Isiolo","Kajiado","Kakamega","Kericho","Kiambu","Kilifi","Kirinyaga","Kisii","Kisumu","Kitui","Kwale","Laikipia","Lamu","Machakos","Makueni","Mandera","Marsabit","Meru","Migori","Mombasa","Murang'a","Nairobi","Nakuru","Nandi","Narok","Nyamira","Nyandarua","Nyeri","Samburu","Siaya","Taita Taveta","Tana River","Tharaka Nithi","Trans Nzoia","Turkana","Uasin Gishu","Vihiga","Wajir","West Pokot"]

# Check required files exist
for fname in ["NDVI_FINAL.csv","Weather_FINAL.csv","Demographic_Crop_Activity_FINAL.csv","master_cleaned_2015_2025.csv"]:
    p = DATA_DIR / fname
    print(f"{'✓' if p.exists() else '✗'} {fname} ({p.stat().st_size/1024:.0f} KB)" if p.exists() else f"✗ {fname} missing")


### Build master


In [ ]:
# Source of truth is master_cleaned_2015_2025.csv (6204 rows, no NA, with _imputed flags)
master_path = DATA_DIR / "master_cleaned_2015_2025.csv"
if master_path.exists():
    df = pd.read_csv(master_path)
    # Normalize for ML: temperature_mean -> temp_mean_c etc. (master uses temp_mean_c already)
    df = df.rename(columns={"temperature_mean":"temp_mean_c","rainfall_sum":"rainfall_mm","soil_moisture_mean":"soil_moisture"})
else:
    # Fallback: join the three source tables (handles fresh clone)
    ndvi = pd.read_csv(DATA_DIR / "NDVI_FINAL.csv")
    weather = pd.read_csv(DATA_DIR / "Weather_FINAL.csv")
    demo = pd.read_csv(DATA_DIR / "Demographic_Crop_Activity_FINAL.csv")
    df = ndvi.merge(weather, on=["county","year","month"], how="inner")
    df = df.merge(demo, on="county", how="left")
    df = df.rename(columns={"temperature_mean":"temp_mean_c","rainfall_sum":"rainfall_mm","soil_moisture_mean":"soil_moisture"})

df["year_month"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)
print(f"Master: {df.shape} | {df['county'].nunique()} counties | {df['year'].min()}-{df['year'].max()} | cols: {list(df.columns)[:6]}...")
display(df.head())


### Verify and save ml_base


In [ ]:
# County check
assert set(COUNTIES) == set(df["county"].unique()), f"County mismatch: missing {set(COUNTIES)-set(df['county'].unique())}"

# Balanced panel: every county same number of months (132 for 2015-2025)
cnt = df.groupby("county").size()
assert cnt.nunique() == 1, f"Unbalanced: {cnt.unique()}"
print(f"Balanced: {cnt.iloc[0]} months/county → {len(df)} rows (47 × {cnt.iloc[0]})")

# Fill any NA (should be 0 for master_cleaned, but keep for robustness)
for col in ["temp_mean_c","rainfall_mm","soil_moisture","ndvi_mean"]:
    if df[col].isna().any():
        df[col] = df.groupby(["year","month"])[col].transform(lambda s: s.fillna(s.mean()))
        df[col] = df[col].fillna(df[col].mean())

assert df[["county","year","month"]].duplicated().sum()==0, "Duplicate county-month"
assert not df.isna().any().any(), f"Still NA: {df.isna().sum().to_dict()}"

# Save ml_base (contract for 01_)
out = DATA_DIR / "ml_base_2015_2025.csv"
df.to_csv(out, index=False)
print(f"✓ Saved {out} ({len(df)} rows, {df.shape[1]} cols) — ready for 01_target_synthesis")
